In [10]:
# ── BLOCK 1: IMPORT LIBRARIES ──────────────────────────────
import pandas as pd
import numpy as np

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")

Libraries loaded successfully
Pandas version: 2.2.3


In [12]:
# ── BLOCK 2: LOAD RAW DATA ─────────────────────────────────
df = pd.read_csv(r"C:\Users\Raymond\nosdra_oilspills_raw.csv")


In [17]:
# # ── BLOCK 3: A LOOK INTO THE DATA ───────────────────────
print("=== COLUMN NAMES ===")
print(df.columns.tolist())

print(f"\n=== DATA TYPES ===")
print(df.dtypes)

print(f"\n=== MISSING VALUES PER COLUMN ===")
print(df.isnull().sum())

print(f"\n=== DUPLICATE ROWS ===")
print(f"Total duplicates: {df.duplicated().sum()}")

In [18]:
# ── BLOCK 4: DROP EMPTY COLUMNS ────────────────
# These 3 columns are entirely null — they add no value
cols_to_drop = ['Unnamed: 22', 'Final lab results date', 'Certificate number']

df.drop(columns=cols_to_drop, inplace=True)

# print(f"Columns dropped: {cols_to_drop}")
# print(f"New shape: {df.shape}")
# print(f"Columns remaining: {len(df.columns)}")

Columns dropped: ['Unnamed: 22', 'Final lab results date', 'Certificate number']
New shape: (1490, 38)
Columns remaining: 38


In [19]:
# ── BLOCK 5: STANDARDIZE COLUMN NAMES ─────────────────────

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# print("Cleaned column names:")
# print(df.columns.tolist())

Cleaned column names:
['id', 'status', 'zonal_office', 'company', 'incident_number', 'incident_date', 'report_date', 'contaminant', 'estimated_quantity', 'quantity_recovered', 'spill_stop_date', 'type_of_facility', 'cause', 'initial_containment_measures', 'site_location_name', 'latitude', 'longitude', 'lga', 'estimated_spill_area', 'spill_area_habitat', 'description_of_impact', 'states_affected', 'form_a_date', 'form_b_date', 'form_c_date', 'jiv_date', 'present_at_jiv', 'cleanup_date', 'cleanup_completed_date', 'cleanup_methods', 'post-cleanup_inspection_date', 'post-impact_assessment_date', 'remediation_start', 'remediation_end', 'remediation_type', 'final_sampling_date', 'certificate_date', 'last_updated_by']


In [28]:
# ── BLOCK 6: DECODE CODED VALUES ──────────────────────────

# Zonal office — NOSDRA regional offices
zonal_map = {
    'ak': 'Akwa Ibom', 'wa': 'Warri',   'ph': 'Port Harcourt',
    'la': 'Lagos',     'ab': 'Abuja',   'en': 'Enugu',
    'ka': 'Kano',      'bn': 'Benin',   'ow': 'Owerri',
    'ib': 'Ibadan',    'by': 'Bayelsa', 'lg': 'Lagos',
    'uy': 'Uyo',       'kd': 'Kaduna'
}
df['zonal_office'] = df['zonal_office'].str.strip().str.lower().map(zonal_map).fillna(df['zonal_office'])

# Contaminant — what was spilled
contaminant_map = {
    'cr': 'Crude Oil',          'co': 'Crude Oil',
    're': 'Refined Product',    'rf': 'Refined Product',
    'gs': 'Gas',                'ga': 'Gas',              'gas': 'Gas',
    'no': 'None/No Spill',      'ch': 'Chemical',
    'water': 'Water',           'non leak': 'None/No Spill',
    'none': 'None/No Spill',    'oil sheen': 'Crude Oil',
    'gli': 'Gas Leak Incident', 'chemical': 'Chemical',
    'gaseous emission fire incident': 'Gas',
}
df['contaminant'] = df['contaminant'].str.strip().str.lower().map(contaminant_map).fillna(df['contaminant'])
df['contaminant'] = df['contaminant'].apply(
    lambda x: 'Other' if isinstance(x, str) and x.lower().startswith('other') else x
)

# Cause — what triggered the spill
cause_map = {
    'sab': 'Sabotage/Third Party',          'cor': 'Corrosion',
    'ome': 'Operational/Maintenance Error', 'eqf': 'Equipment Failure',
    'unk': 'Unknown',                       'nat': 'Natural Disaster',
    'tpi': 'Third Party Interference'
}
df['cause'] = df['cause'].str.strip().str.lower().map(cause_map).fillna(df['cause'])

# Type of facility — infrastructure where spill occurred
facility_map = {
    'fp': 'FPSO',            'fl': 'Flowline',        'wh': 'Wellhead',
    'pp': 'Pipeline',        'pl': 'Pipeline',         'tk': 'Tank',
    'mn': 'Manifold',        'mf': 'Manifold',         'tr': 'Terminal',
    'rf': 'Refinery',        'lb': 'Loading Buoy',     'fs': 'Flow Station',
    'st': 'Storage Tank',    'gl': 'Gas Line',          'tf': 'Tank Farm',
    'dl': 'Delivery Line',   'cp': 'Central Processing Facility',
    'ps': 'Pumping Station', 'fd': 'Flow Dam',          'rg': 'Riser',
    'pipeline': 'Pipeline',  'flow line': 'Flowline',   'pipiline': 'Pipeline',
    'oil well': 'Wellhead',  'bulkline': 'Bulkline',    'delivery line': 'Delivery Line',
}
df['type_of_facility'] = df['type_of_facility'].str.strip().str.lower().map(facility_map).fillna(df['type_of_facility'])
df['type_of_facility'] = df['type_of_facility'].apply(
    lambda x: 'Other' if isinstance(x, str) and x.lower().startswith('other') else x
)

# Spill area habitat — environment the spill entered
habitat_map = {
    'la': 'Land',            'sw': 'Swamp',
    'of': 'Offshore',        'iw': 'Inshore Water',
    'sh': 'Shoreline',       'cr': 'Creek',
    'ss': 'Shoreline',       'co': 'Coastal',
    'ns': 'Near Shore',      'la,ss': 'Land/Shoreline',
    'la,ns': 'Land/Near Shore',  'la,sw': 'Land/Swamp',
    'la,iw': 'Land/Inshore Water', 'sw,la': 'Swamp/Land',
    'sw,iw': 'Swamp/Inshore Water', 'sw,ss': 'Swamp/Shoreline',
    'iw,sw': 'Inshore Water/Swamp', 'iw,la': 'Inshore Water/Land',
    'iw,ns': 'Inshore Water/Near Shore', 'ss,la': 'Shoreline/Land',
    'co,ns': 'Coastal/Near Shore', 'ss,sw': 'Shoreline/Swamp',
}
df['spill_area_habitat'] = df['spill_area_habitat'].str.strip().str.lower().map(habitat_map).fillna(df['spill_area_habitat'])

# States affected
state_map = {
    'ON': 'Ondo',      'DE': 'Delta',      'RI': 'Rivers',
    'BY': 'Bayelsa',   'AK': 'Akwa Ibom',  'CR': 'Cross River',
    'IM': 'Imo',       'AB': 'Abia',       'ED': 'Edo',
    'LA': 'Lagos'
}
df['states_affected'] = df['states_affected'].str.strip().map(state_map).fillna(df['states_affected'])

# # Verify all columns
# print("✓ Zonal office:\n", df['zonal_office'].value_counts().to_string())
# print("\n✓ Contaminant:\n", df['contaminant'].value_counts().to_string())
# print("\n✓ Cause:\n", df['cause'].value_counts().to_string())
# print("\n✓ Facility:\n", df['type_of_facility'].value_counts().to_string())
# print("\n✓ Habitat:\n", df['spill_area_habitat'].value_counts().to_string())
# print("\n✓ States:\n", df['states_affected'].value_counts().to_string())

In [27]:
# ── BLOCK 7: FIX REMAINING CAUSE AND STATES VALUES ───────

# Standardize all "other:" variants and unknown codes
df['cause'] = df['cause'].apply(
    lambda x: 'Other/Unspecified' if isinstance(x, str) and x.lower().startswith('other') else x
)
df['cause'] = df['cause'].replace({
    'ytd': 'Other/Unspecified'
})

# Handle combined and unrecognized codes
states_extra = {
    'RI,IM': 'Rivers/Imo',
    'ED,RI': 'Edo/Rivers',
    'AN':    'Anambra',
    'KD':    'Kaduna',
    'EK':    'Ekiti',
    'ZA':    'Zamfara',
}
df['states_affected'] = df['states_affected'].replace(states_extra)

# # Verify
# print("✓ Cause:\n", df['cause'].value_counts().to_string())
# print("\n✓ States:\n", df['states_affected'].value_counts().to_string())

In [26]:
# ── BLOCK 8: FIX DATA TYPES ───────────────────────────────

# Date columns
date_cols = [
    'incident_date', 'report_date', 'spill_stop_date',
    'form_a_date', 'form_b_date', 'form_c_date',
    'jiv_date', 'cleanup_date', 'cleanup_completed_date',
    'post-cleanup_inspection_date', 'post-impact_assessment_date',
    'remediation_start', 'remediation_end',
    'final_sampling_date', 'certificate_date'
]
df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')

# Numeric columns
num_cols = ['estimated_quantity', 'quantity_recovered', 'latitude', 'longitude']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

# # Verify
# print("=== DATA TYPES ===")
# print(df.dtypes)
# print(f"\nDate range: {df['incident_date'].min()} to {df['incident_date'].max()}")
# print(f"Quantity range: {df['estimated_quantity'].min()} to {df['estimated_quantity'].max()} barrels")

In [30]:
# ── BLOCK 9: CREATE DERIVED COLUMNS ───────────────────────
# These columns don't exist in the raw data — we engineer them
# to make time-based and severity analysis possible

df['year']         = df['incident_date'].dt.year
df['month']        = df['incident_date'].dt.month
df['month_name']   = df['incident_date'].dt.strftime('%B')

df['spill_severity'] = pd.cut(
    df['estimated_quantity'],
    bins=[0, 1, 10, 100, float('inf')],
    labels=['Minor (<1 bbl)', 'Low (1-10 bbl)', 'Medium (10-100 bbl)', 'Major (>100 bbl)']
)

df['quantity_recovered_pct'] = (
    (df['quantity_recovered'] / df['estimated_quantity']) * 100
).clip(0, 100).round(2)

df['jiv_conducted']    = df['jiv_date'].notna().map({True: 'Yes', False: 'No'})
df['days_to_cleanup']  = (df['cleanup_date'] - df['incident_date']).dt.days

# # Verify
# print("=== DERIVED COLUMNS ===")
# print(f"\nYear distribution:\n{df['year'].value_counts().sort_index()}")
# print(f"\nSpill severity:\n{df['spill_severity'].value_counts()}")
# print(f"\nJIV conducted:\n{df['jiv_conducted'].value_counts()}")
# print(f"\nAvg days to cleanup: {df['days_to_cleanup'].mean():.1f} days")

In [32]:
# ── BLOCK 10: STANDARDIZE TEXT FIELDS ──────────────────────
# Consistent casing ensures GROUP BY in SQL works correctly
# "naoc" and "NAOC" and "Naoc" would otherwise count as three separate companies

text_cols = ['company', 'lga', 'status', 'site_location_name']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

# # Verify
# print("=== SAMPLE TEXT FIELDS AFTER STANDARDIZATION ===")
# print(f"\nTop 10 companies:\n{df['company'].value_counts().head(10)}")
# print(f"\nStatus values:\n{df['status'].value_counts()}")

In [33]:
# ── BLOCK 11: FIX ACRONYM COMPANY NAMES ──────────────────

acronym_fix = {
    'Naoc':  'NAOC',   # Nigerian Agip Oil Company
    'Spdc':  'SPDC',   # Shell Petroleum Development Company
    'Nepl':  'NEPL',   # Nigerian Petroleum Development Company
    'Mpn':   'MPN',    # Mobil Producing Nigeria
    'Npdc':  'NPDC',   # Nigerian Petroleum Development Company
    'Nnpc':  'NNPC',   # Nigerian National Petroleum Corporation
    'Tepng': 'TEPNG',  # Total E&P Nigeria
    'Snepco':'SNEPCO', # Shell Nigeria Exploration and Production
}

df['company'] = df['company'].replace(acronym_fix)

print("✓ Company names fixed:")
print(df['company'].value_counts().head(10))

✓ Company names fixed:
company
NAOC                      431
SPDC                      301
Oando                     213
Renaissance               103
Seplat                     70
Heirs Energies Limited     70
NEPL                       67
Heritage                   57
MPN                        50
Chevron                    21
Name: count, dtype: int64


In [34]:
# ── BLOCK 12: EXPORT CLEANED DATA ─────────────────────────

output_path = r"C:\Users\Raymond\nosdra_oilspills_cleaned.csv"
df.to_csv(output_path, index=False)

print("=== FINAL CLEANING SUMMARY ===")
print(f"Raw shape:     (1490, 41)")
print(f"Cleaned shape: {df.shape}")
print(f"\nCleaning steps applied:")
print("  ✓ Dropped 3 empty columns")
print("  ✓ Standardized column names")
print("  ✓ Decoded coded values — zonal office, contaminant, cause, facility, habitat, states")
print("  ✓ Fixed data types — 15 date columns, 4 numeric columns")
print("  ✓ Engineered 7 derived columns — year, month, severity, recovery %, JIV flag, days to cleanup")
print("  ✓ Standardized text fields and corrected acronyms")
print(f"\n✓ Cleaned data saved to: {output_path}")

=== FINAL CLEANING SUMMARY ===
Raw shape:     (1490, 41)
Cleaned shape: (1490, 45)

Cleaning steps applied:
  ✓ Dropped 3 empty columns
  ✓ Standardized column names
  ✓ Decoded coded values — zonal office, contaminant, cause, facility, habitat, states
  ✓ Fixed data types — 15 date columns, 4 numeric columns
  ✓ Engineered 7 derived columns — year, month, severity, recovery %, JIV flag, days to cleanup
  ✓ Standardized text fields and corrected acronyms

✓ Cleaned data saved to: C:\Users\Raymond\nosdra_oilspills_cleaned.csv


In [35]:
# ── BLOCK 13: FIX COLUMN NAMES WITH DASHES ───────────────
# PostgreSQL treats dashes as minus signs — rename to underscores

df.rename(columns={
    'post-cleanup_inspection_date': 'post_cleanup_inspection_date',
    'post-impact_assessment_date':  'post_impact_assessment_date'
}, inplace=True)

# Re-export with fixed column names
output_path = r"C:\Users\Raymond\nosdra_oilspills_cleaned.csv"
df.to_csv(output_path, index=False)
print("✓ File re-exported with fixed column names")
print(f"Columns: {df.columns.tolist()}")

✓ File re-exported with fixed column names
Columns: ['id', 'status', 'zonal_office', 'company', 'incident_number', 'incident_date', 'report_date', 'contaminant', 'estimated_quantity', 'quantity_recovered', 'spill_stop_date', 'type_of_facility', 'cause', 'initial_containment_measures', 'site_location_name', 'latitude', 'longitude', 'lga', 'estimated_spill_area', 'spill_area_habitat', 'description_of_impact', 'states_affected', 'form_a_date', 'form_b_date', 'form_c_date', 'jiv_date', 'present_at_jiv', 'cleanup_date', 'cleanup_completed_date', 'cleanup_methods', 'post_cleanup_inspection_date', 'post_impact_assessment_date', 'remediation_start', 'remediation_end', 'remediation_type', 'final_sampling_date', 'certificate_date', 'last_updated_by', 'year', 'month', 'month_name', 'spill_severity', 'quantity_recovered_pct', 'jiv_conducted', 'days_to_cleanup']


In [36]:
!pip install sqlalchemy psycopg2-binary